In [8]:
from pathlib import Path

import numpy as np
import pandas as pd


# Названия файлов удобно менять здесь
input_filename = "10exp.csv"
vle_filename = "vle-boiling-point.csv"
output_filename = "exp10-with-T.csv"

EXP_DATA_DIR = Path("VLE/data")
VLE_DATA_DIR = Path("Experiment-data-processing/vle-data")


# Молярные массы как в текущей обработке VLE-графиков
M_WATER = 18.0
M_ETHANOL = 46.0
M_HEXANE = 86.0


def find_project_root(start=None):
    """Находит корень проекта независимо от того, откуда запущен notebook."""
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (
            (path / EXP_DATA_DIR / input_filename).exists()
            and (path / VLE_DATA_DIR / vle_filename).exists()
        ):
            return path
    raise FileNotFoundError(f"Не удалось найти корень проекта с {EXP_DATA_DIR / input_filename}")


PROJECT_ROOT = find_project_root()
EXP10_PATH = PROJECT_ROOT / EXP_DATA_DIR / input_filename
VLE_PATH = PROJECT_ROOT / VLE_DATA_DIR / vle_filename
OUTPUT_PATH = PROJECT_ROOT / EXP_DATA_DIR / output_filename


def mass_percent_to_mole_fraction(water_pct, ethanol_pct, hexane_pct):
    """Переводит массовые проценты H2O/EtOH/hexane в молярные доли."""
    n_water = water_pct / M_WATER
    n_ethanol = ethanol_pct / M_ETHANOL
    n_hexane = hexane_pct / M_HEXANE
    n_sum = n_water + n_ethanol + n_hexane

    if n_sum <= 0 or np.isnan(n_sum):
        return np.nan, np.nan, np.nan

    return n_water / n_sum, n_ethanol / n_sum, n_hexane / n_sum


def read_vle_reference(path):
    """Читает VLE-таблицу и убирает строку с единицами измерения."""
    df = pd.read_csv(path)
    required = ["Temperature", "x1", "x2", "y1", "y2"]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"В VLE CSV нет обязательного столбца {col!r}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=required).reset_index(drop=True)
    df["x3"] = 1.0 - df["x1"] - df["x2"]
    df["y3"] = 1.0 - df["y1"] - df["y2"]
    return df


def add_nearest_temperatures(exp_path=EXP10_PATH, vle_path=VLE_PATH, filename=OUTPUT_PATH):
    # 10exp.csv называется CSV, но фактически разделён табуляцией.
    # sep=None здесь плохо подходит: запятые в заголовках вида "x_вода,%" мешают autodetect.
    exp_df = pd.read_csv(exp_path, sep="\t")
    vle_df = read_vle_reference(vle_path)

    exp_cols = [
        "x_вода,%",
        "x_спирт,%",
        "x_гексан,%",
        "y_вода,%",
        "y_спирт,%",
        "y_гексан,%",
    ]

    missing_cols = [col for col in exp_cols if col not in exp_df.columns]
    if missing_cols:
        raise ValueError(f"В 10exp.csv нет обязательных столбцов: {missing_cols}")

    for col in exp_cols:
        exp_df[col] = pd.to_numeric(exp_df[col], errors="coerce")

    x_mol = np.array(
        [mass_percent_to_mole_fraction(row["x_вода,%"], row["x_спирт,%"], row["x_гексан,%"])
         for _, row in exp_df.iterrows()]
    )
    y_mol = np.array(
        [mass_percent_to_mole_fraction(row["y_вода,%"], row["y_спирт,%"], row["y_гексан,%"])
         for _, row in exp_df.iterrows()]
    )

    exp_points_mol = np.column_stack([x_mol, y_mol])
    vle_points_mol = vle_df[["x1", "x2", "x3", "y1", "y2", "y3"]].to_numpy()

    nearest_temperatures = []
    nearest_distances = []

    for point in exp_points_mol:
        if np.isnan(point).any():
            nearest_temperatures.append(np.nan)
            nearest_distances.append(np.nan)
            continue

        distances = np.linalg.norm(vle_points_mol - point, axis=1)
        nearest_idx = int(np.argmin(distances))
        nearest_temperatures.append(vle_df.loc[nearest_idx, "Temperature"])
        nearest_distances.append(distances[nearest_idx])

    result_df = exp_df.copy()
    result_df["T кипения, С"] = np.round(nearest_temperatures, 2)

    # Если нужно проверить качество подбора, можно временно раскомментировать строку ниже.
    # result_df["nearest_distance_mol"] = nearest_distances

    result_df.to_csv(filename, sep="\t", index=False)
    print(f"Записано строк: {len(result_df)}")
    print(f"Файл: {filename}")
    return result_df


exp10_with_t = add_nearest_temperatures()
exp10_with_t


Записано строк: 10
Файл: /Users/artemis/Documents/S26S/Project-Labs-S26S/VLE/data/exp10-with-T.csv


,№ опыта,"x_вода,%","x_спирт,%","x_гексан,%","y_вода,%","y_спирт,%","y_гексан,%","T кипения, С"
0,1,3.75,15.45,80.80,2.03,15.02,82.95,56.92
1,2,10.18,33.06,56.76,6.82,17.14,76.04,56.92
2,3,11.84,47.52,40.64,7.09,15.88,77.03,56.89
3,4,1.59,38.21,60.20,6.37,14.56,79.08,57.17
4,5,7.00,49.77,43.22,14.89,26.71,58.40,62.87
5,6,14.08,51.81,34.11,4.27,17.15,78.58,56.89
6,7,10.73,11.48,77.80,3.91,14.08,82.01,57.34
7,8,29.70,5.86,64.44,6.42,7.40,86.19,59.19
8,9,23.49,71.36,5.15,4.19,95.81,0.00,78.79
9,10,52.85,42.18,4.97,28.50,71.50,0.00,83.06
